# Train and prune a traffic-sign classifier
**SYDE 411 · Group 28**

How does a classifier change when two pruning methods have to meet the same weight budget? Start with one dense CNN, prune it two ways, inspect the validation results, then repeat the experiment across three initializations and evaluate the frozen models on the official GTSRB test set.

Run the cells in order with the Python 3.11 environment from the [README](../README.md). **Run All** executes the full project. For a shorter first session, stop after the first validation plot (step 5). The [results notebook](results.ipynb) contains the original report measurements.

## 1. Set up a local run

Choose a run name. On the first pass, the notebook creates a new directory under `notebook-runs/`. Keep the name to resume after a break, or change it to start from new weights. Use a different name if you change the training code or settings. The original checkpoints live in the repository’s separate `runs/` directory.

In [ ]:
import csv
import os
import shutil
import subprocess
import sys
from pathlib import Path
from statistics import mean

import torch
from IPython.display import Image, Markdown, display

run_name = "group-reproduction"
if not run_name or not all(c.isalnum() or c in "-_" for c in run_name):
    raise ValueError(
        "Choose a run name using letters, numbers, hyphens or underscores."
    )

repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "grid.py").is_file())
work = repo / "notebook-runs" / run_name
data = work / "data"
data.mkdir(parents=True, exist_ok=True)
env = {**os.environ, "SYDE411_WORKSPACE": str(work), "PYTHONUNBUFFERED": "1"}
pair = work / "runs" / "comparison" / "seed411"
print("Run folder:", work.relative_to(repo))
print("Training device:", "Apple MPS" if torch.backends.mps.is_available() else "CPU")

In [ ]:
def run(script, *args, compact=False):
    """Run a project script with this kernel’s Python and stream its progress."""
    command = [sys.executable, str(repo / script), *map(str, args)]
    with subprocess.Popen(
        command,
        cwd=repo,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ) as process:
        tail = []
        for line in process.stdout:
            line = line.rstrip()
            tail = (tail + [line])[-12:]
            if not compact or line.startswith("Verified"):
                print(line, flush=True)
        if process.wait():
            raise RuntimeError(f"{script} failed:\n" + "\n".join(tail))


def read_table(path):
    with path.open(newline="") as source:
        return list(csv.DictReader(source))


def train_first_pair(method, *, epochs, lr, sparsity=None):
    stem = method if sparsity is None else f"{method}{sparsity}"
    if (pair / f"{stem}.pt").is_file():
        print(f"Reusing this run's completed {stem} checkpoint.")
        return
    args = [
        method,
        "--seed",
        411,
        "--tag",
        "comparison",
        "--epochs",
        epochs,
        "--lr",
        lr,
    ]
    if sparsity is not None:
        args.extend(["--sparsity", sparsity])
    run("experiment.py", *args)

## 2. Prepare and inspect the data

The first run fetches the official archives as needed. If the archives are already in the checkout, the notebook can reuse them. The preparation script verifies the training archive, crops the sign regions to 32 × 32 RGB and splits by physical sign track. Frames of the same sign must stay together.

In [ ]:
for archive in (
    "GTSRB_Final_Training_Images.zip",
    "GTSRB_Final_Test_Images.zip",
    "GTSRB_Final_Test_GT.zip",
):
    cached = repo / "data" / archive
    if cached.is_file() and not (data / archive).exists():
        partial = (data / archive).with_suffix(".part")
        shutil.copyfile(cached, partial)
        partial.replace(data / archive)

if not (data / "prepared.npz").is_file():
    run("prepare.py")
manifest = read_table(data / "manifest.csv")
tracks = {
    split: {row["track"] for row in manifest if row["partition"] == split}
    for split in ("train", "validation")
}
assert tracks["train"].isdisjoint(tracks["validation"])
assert (data / "manifest.csv").read_bytes() == (repo / "data/manifest.csv").read_bytes()
for split, sign_tracks in tracks.items():
    count = sum(row["partition"] == split for row in manifest)
    print(
        f"{split.capitalize()}: {count:,} images from {len(sign_tracks):,} sign tracks."
    )

## 3. Train the dense baseline

Train the three-convolution CNN on all 43 classes for 12 epochs with Adam. Each progress line gives the checkpoint name, epoch, update count, validation accuracy and cumulative training seconds. The convolutions have 32, 64 and 128 output channels. Global average pooling and a linear layer produce one score per class. Both pruning methods below reload these same trained weights. [CNN and training loop](../experiment.py)

In [ ]:
train_first_pair("dense", epochs=12, lr=0.003)
baseline = read_table(pair / "dense.csv")[-1]
print(f"Dense validation accuracy: {float(baseline['accuracy']):.2%}")

## 4. Prune the same model two ways

**Global magnitude pruning.** Keep the 49,264 weights with the largest absolute values, then train for four epochs with the other weights fixed at zero. Biases are exempt from the cap. After each optimizer step, the training loop reapplies the mask so the discarded weights stay zero.

In [ ]:
train_first_pair("magnitude", epochs=4, lr=0.001, sparsity=50)

**ADMM.** Alternate training with projection onto the same weight budget for three epochs (penalty ρ = 0.001). In each of the three outer iterations, the training loop updates the network weights, projects the shifted weights onto the global top-49,264 constraint, and updates the dual variable. Fix the final mask and retrain for one more epoch. Both methods see the same minibatch order and get 984 gradient updates.

In [ ]:
train_first_pair("admm", epochs=4, lr=0.001, sparsity=50)

## 5. Compare on validation data

The plot shows how loss changes and which layers retain weights. The checks below compare the update counts and final weight counts of the two pruned models. This completes the single-seed walkthrough.

In [ ]:
run("summarize.py", pair, "--sparsity", 50, compact=True)
summary = read_table(pair / "comparison50.csv")
for row in summary:
    if row["method"] in ("Magnitude", "ADMM"):
        assert int(row["updates"]) == 984 and int(row["nonzeros"]) <= 49_264
    print(
        f"{row['method']:10} {float(row['validation_accuracy']):6.2%} validation accuracy"
        f"   {int(row['nonzeros']):6,} nonzero weights"
    )
display(Image(filename=str(pair / "comparison50.png"), width=780))

## 6. Complete the experiment grid

Repeat the comparison at 50%, 80% and 90% sparsity for seeds 411, 412 and 413. The grid resumes the three models just trained and fits the remaining 18. Each pruning pair shares its seed’s dense checkpoint.

In [ ]:
run("grid.py", compact=True)

## 7. Test sensitivity on the validation split

Starting from seed 411’s dense model at an 80% pruning target, halve or double one of two parameters at a time: the ADMM penalty or the learning rate. The original 80% run supplies the reference setting.

In [ ]:
run("sensitivity.py", compact=True)
for tag in ("comparison", "rho_half", "rho_double", "rate_half", "rate_double"):
    final = read_table(work / "runs" / tag / "seed411" / "admm80.csv")[-1]
    print(f"{tag:12} validation cross-entropy {float(final['ce']):.3f}")

## 8. Evaluate the frozen models

Evaluate the main grid on the 12,630 official test images. Compare your three-seed mean accuracy with the original report. Hardware and library differences can affect the values.

In [ ]:
run("evaluate.py")
results = work / "results"
fresh = read_table(results / "test_results.csv")
reference = read_table(repo / "results" / "test_results.csv")
assert len(fresh) == 21 and len(read_table(results / "paired_differences.csv")) == 9


def mean_accuracy(rows, method, sparsity):
    values = [
        float(r["accuracy"])
        for r in rows
        if r["method"] == method and int(r["sparsity"]) == sparsity
    ]
    assert len(values) == 3
    return mean(values)


lines = [
    "| Model | Weights pruned | Your run | Original report |",
    "| :--- | ---: | ---: | ---: |",
]
settings = [("dense", 0)] + [
    (m, s) for s in (50, 80, 90) for m in ("magnitude", "admm")
]
for method, sparsity in settings:
    label = {"dense": "Dense", "magnitude": "Magnitude", "admm": "ADMM"}[method]
    lines.append(
        f"| {label} | {sparsity}% | {mean_accuracy(fresh, method, sparsity):.2%} "
        f"| {mean_accuracy(reference, method, sparsity):.2%} |"
    )
display(Markdown("\n".join(lines)))

## 9. Check optimality and redraw the figures

The audit tests whether moving the output biases can decrease the full training objective while preserving the weight budget. A positive decrease demonstrates a feasible improving direction at that checkpoint. The two figures use your measurements, including your machine’s elapsed training times.

In [ ]:
run("audit_objective.py")
audit = read_table(results / "training_objective_audit.csv")
assert len(audit) == 21
improved = sum(float(row["feasible_decrease"]) > 0 for row in audit)
print(
    f"A feasible bias step decreased the objective in {improved} of {len(audit)} checkpoints."
)

run("figures.py")
for filename in ("test_comparison.png", "solver_behavior.png"):
    display(Image(filename=str(results / filename), width=760))
print("Your result tables and figures:", results.relative_to(repo))